In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import os
import pandas as pd
import torch

from src.dataloader import GraphTimeSeriesLoader, GraphTimeSeriesDataset

%load_ext autoreload
%autoreload 2

In [3]:
processed_data_dir = "data/datos_procesados"
dataset_filenames = {
    "buildings": f"{processed_data_dir}/buildings/roads_buildings.csv",
    "bus_stops": f"{processed_data_dir}/bus_stops/bus_stops_per_road.parquet",
    "events": f"{processed_data_dir}/events/events_dataset.csv",
    "incidents": f"{processed_data_dir}/incidents/incidents_dataset.csv",
    "irregularities": [f"{processed_data_dir}/irregularidades_procesadas/" + f for f in os.listdir(f"{processed_data_dir}/irregularidades_procesadas/") if f.endswith('.parquet')],
    # "coi_points": "pmts_processed/coi_puntos_dataset.parquet",
    # "coi_parks": "pmts_processed/coi_parques_dataset.parquet",
    "coi_points": [f"{processed_data_dir}/pmts_processed/coi_puntos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_puntos/") if f.endswith('.parquet')],
    "coi_tracts": [f"{processed_data_dir}/pmts_processed/coi_tramos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_tramos/") if f.endswith('.parquet')],
    "road_state": f"{processed_data_dir}/road_state/road_state_dateset.parquet", # TODO: Should be 'dataset' no 'dateset'
}
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
imputation_parquet_filename = f"{processed_data_dir}/neighbor_features.parquet"

In [4]:
loader = GraphTimeSeriesLoader(
    graph_pkl_filename=graph_filename,
    buildings_csv_filename=dataset_filenames["buildings"],
    bus_stops_parquet_filename=dataset_filenames["bus_stops"],
    events_csv_filename=dataset_filenames["events"],
    incidents_csv_filename=dataset_filenames["incidents"],
    road_state_parquet_filename=dataset_filenames["road_state"],
    irregularities_filenames=dataset_filenames["irregularities"],
    coi_points_filenames=dataset_filenames["coi_points"],
    coi_tracts_filenames=dataset_filenames["coi_tracts"],
    default_speeds_parquet_filename=imputation_parquet_filename,
)

In [5]:
# incident_set = set([tuple(l) for l in loader.incidents_df[["road_id", "timeframe"]].values])
# event_set = set([tuple(l) for l in loader.events_df[["road_id", "timeframe"]].values])
# common_tuples = incident_set.intersection(event_set)
# list(common_tuples)[:5]

In [6]:
# # Convert the nx graph to a pandas DataFrame to check that everything is loaded correctly
# all_nodes_df = pd.DataFrame.from_dict(dict(loader.base_graph.nodes(data=True)), orient='index')
# all_nodes_df

In [5]:
# Example usage of the loader to load data for specific nodes and time windows
nodes_to_load = ["9004322_TF.1", "50008506_TF.1", "12003262_FT.1"]
time_windows_to_load = [
    ('2024-09-29 23:00:00', 4, 5),
    ('2024-08-24 15:30:00', 2, 0),
    ('2024-02-24 01:30:00', 0, 3)
]

# Load the data
(static_features, temporal_features), target = loader.load_data(
    nodes_to_load, time_windows_to_load
)


--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (51, 112) ---


In [6]:
static_features

,road_id,mvi_codigo,mvi_ccalzad,mvi_civ,mvi_ccat,mvi_tcla,mvi_tipo,mvi_nombre,mvi_naltern,mvi_nprinci,...,warehouse,water_tank,water_tankmm,yes,yesmm,num_stops,num_stops_m,num_stops_s,mean_distance_meters,median_distance_meters
0,9004322_TF.1,116225,228531.0,9004322,9004150,1,AC,AVENIDA LUIS CARLOS GALAN,,0240000000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,50008506_TF.1,51675,272273.0,50008506,12000648,1,AC,AVENIDA JOSE CELESTINO MUTIS,,0630000000,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
2,12003262_FT.1,54323,902937.0,12003262,12001715,1,AK,AVENIDA CIUDAD DE QUITO,,0300000000,...,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
temporal_features.head(5)

,road_id,timeframe,aglomeracion,emergencia,manifestacion,incident_type_703,incident_type_715,incident_type_802,incident_type_901,incident_type_910,...,n_thumbs_up,seconds,cause_type,cause_type_0,cause_type_accident,cause_type_hazard,cause_type_road_closed,severity,coi_puntos,coi_tramo
0,9004322_TF.1,2024-02-24 00:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,9004322_TF.1,2024-02-24 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,9004322_TF.1,2024-02-24 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,9004322_TF.1,2024-02-24 01:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,9004322_TF.1,2024-08-24 15:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
from torch.utils.data import DataLoader

# Create an instance of the custom dataset
train_dataset = GraphTimeSeriesDataset(loader=loader, node_ids=nodes_to_load, time_windows=time_windows_to_load)

# Create the DataLoader
# This will handle batching, shuffling, and parallel data loading.
batch_size = 2
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # Shuffle the data at the beginning of each epoch
    num_workers=0   # Set to > 0 to use multiprocessing for data loading
)

# Iterate over the DataLoader in the training loop
num_epochs = 3
for epoch in range(num_epochs):
    print(f"--- Epoch {epoch+1}/{num_epochs} ---")
    for i, (static_batch, temporal_batch, target_batch) in enumerate(train_dataloader):
        print(f"  Batch {i+1}:")
        print(f"    Static data shape: {static_batch.shape}")
        print(f"    Temporal data shape: {temporal_batch.shape}")
        print(f"    Target data shape: {target_batch.shape}")
        print("-" * 20)

        # Here the batches should be fed to the GNN model for training
        # model.train()
        # optimizer.zero_grad()
        # output = model(static_batch, temporal_batch)
        # loss = criterion(output, target_batch)
        # loss.backward()
        # optimizer.step()

--- Epoch 1/3 ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (12, 112) ---


/Users/martinsevilla/Downloads/congestion-bogota/src/dataloader.py:584: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_temporal_df[col] = 0
/Users/martinsevilla/Downloads/congestion-bogota/src/dataloader.py:584: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_temporal_df[col] = 0
/Users/martinsevilla/Downloads/congestion-bogota/src/dataloader.py:584: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all co

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.